In [1]:
import pandas as pd

household_power = pd.read_csv('household_power.csv')
household_power.head()

,time,sensor.power_load_no_var_loads,ac_w,water_w,total_w
0,2025-06-01 00:00:00,503.406806,1800.0,0.0,2303.406806
1,2025-06-01 00:30:00,502.879320,1800.0,0.0,2302.879320
2,2025-06-01 01:00:00,514.847412,1800.0,0.0,2314.847412
3,2025-06-01 01:30:00,476.294995,1800.0,0.0,2276.294995
4,2025-06-01 02:00:00,516.189519,1800.0,0.0,2316.189519


In [2]:
household_power_df = household_power[['time', 'sensor.power_load_no_var_loads','total_w', 'ac_w', 'water_w']]
# Add p_load column as sum of base_w, ac_w, and water_w
household_power_df['p_load'] = household_power_df['total_w'] 

# Ensure 'time' is a datetime column
household_power_df['time'] = pd.to_datetime(household_power_df['time'])

# def get_unit_load_cost(ts):
#     hour = ts.hour + ts.minute/60
#     if (7 <= hour < 9) or (17.5 <= hour < 20.5):
#         return 0.1907
#     else:
#         return 0.1419

# # Map over the 'time' column, not the index
# household_power_df['unit_load_cost'] = household_power_df['time'].map(get_unit_load_cost)
# # Add unit_prod_price column: always 0.1419
# household_power_df['unit_prod_price'] = 0.1419

# household_power_df[['time','p_load', 'ac_w', 'water_w','unit_load_cost', 'unit_prod_price']].head()

C:\Users\26293\AppData\Local\Temp\ipykernel_35764\1393683611.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  household_power_df['p_load'] = household_power_df['total_w']
C:\Users\26293\AppData\Local\Temp\ipykernel_35764\1393683611.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  household_power_df['time'] = pd.to_datetime(household_power_df['time'])


In [3]:
optimal_df = pd.read_csv('opt_res_latest.csv')
optimal_df.head()

,timestamp,P_PV,P_Load,P_deferrable0,P_deferrable1,P_grid_pos,P_grid_neg,P_grid,unit_load_cost,unit_prod_price,cost_profit,cost_fun_profit,optim_status
0,2025-07-08 17:30:00+08:00,1043.478828,503.406806,0.0,0.0,0.00000,-540.072020,-540.072020,0.1907,0.1419,0.038318,0.038318,Optimal
1,2025-07-08 18:00:00+08:00,587.368248,502.879320,0.0,0.0,0.00000,-84.488928,-84.488928,0.1907,0.1419,0.005994,0.005994,Optimal
2,2025-07-08 18:30:00+08:00,215.377384,514.847412,0.0,0.0,299.47003,0.000000,299.470030,0.1907,0.1419,-0.028554,-0.028554,Optimal
3,2025-07-08 19:00:00+08:00,0.000000,476.294995,0.0,0.0,476.29499,0.000000,476.294990,0.1907,0.1419,-0.045415,-0.045415,Optimal
4,2025-07-08 19:30:00+08:00,0.000000,516.189519,0.0,0.0,516.18952,0.000000,516.189520,0.1907,0.1419,-0.049219,-0.049219,Optimal


In [4]:
if 'P_grid_neg' in optimal_df.columns:
    total_prod_revenue = (0.5*optimal_df['unit_prod_price'] * 0.001* (-optimal_df['P_grid_neg'])).sum()
    print('Optimal revenue of a day (unit_prod_price * P_grid_neg):', total_prod_revenue)
else:
    print('P_grid_neg column not found in optimal_df')

for col in ['P_grid_pos']:
    if col not in optimal_df.columns:
        optimal_df[col] = 0  # fallback if missing
if 'unit_load_cost' not in optimal_df.columns:
    optimal_df['unit_load_cost'] = 0.1419  # fallback if not present
total_load_cost = 0.5*(0.001*(optimal_df['P_grid_pos']) * optimal_df['unit_load_cost']).sum()
print('Optimal cost of a day ((P_grid_pos) * unit_load_cost):', total_load_cost)

optimal_cost = total_load_cost - total_prod_revenue
print('Optimal Total cost:', optimal_cost)

Optimal revenue of a day (unit_prod_price * P_grid_neg): 0.41348812400082
Optimal cost of a day ((P_grid_pos) * unit_load_cost): 4.28656026177121
Optimal Total cost: 3.87307213777039


In [5]:
# Select rows from household_power_df between 2025-07-07 16:30:00 and 2025-07-08 16:00:00
start_time = pd.Timestamp('2025-07-07 17:30:00')
end_time = pd.Timestamp('2025-07-08 17:00:00')
selected_df = household_power_df[(household_power_df['time'] >= start_time) & (household_power_df['time'] <= end_time)]
selected_df.head()

,time,sensor.power_load_no_var_loads,total_w,ac_w,water_w,p_load
1763,2025-07-07 17:30:00,713.086690,713.086690,0.0,0.0,713.086690
1764,2025-07-07 18:00:00,689.707076,2489.707076,1800.0,0.0,2489.707076
1765,2025-07-07 18:30:00,728.783974,2528.783974,1800.0,0.0,2528.783974
1766,2025-07-07 19:00:00,1995.163503,6195.163503,1800.0,2400.0,6195.163503
1767,2025-07-07 19:30:00,1864.374669,6064.374669,1800.0,2400.0,6064.374669


In [ ]:
# Merge p_load from household_power_df with P_PV, unit_load_cost, unit_prod_price from optimal_df
# Assumes both DataFrames are aligned by row (same time order)
combined_df = pd.DataFrame({
    'time': selected_df['time'].values,
    'p_load': selected_df['p_load'].values,
    'P_PV': optimal_df['P_PV'].values if 'P_PV' in optimal_df.columns else 0,
    'unit_load_cost': optimal_df['unit_load_cost'].values,
    'unit_prod_price': optimal_df['unit_prod_price'].values if 'unit_prod_price' in optimal_df.columns else 0.1419,
})

# Compute P_grid
combined_df['P_grid'] = combined_df['p_load'] - combined_df['P_PV']

# Create P_grid_pos and P_grid_neg columns
combined_df['P_grid_pos'] = combined_df['P_grid'].apply(lambda x: x if x > 0 else 0)
combined_df['P_grid_neg'] = combined_df['P_grid'].apply(lambda x: x if x < 0 else 0)

combined_df.head()

In [ ]:
combined_df.to_csv('combined_df.csv', index=False)

In [ ]:
origin_revenue = (0.5*combined_df['unit_prod_price'] * 0.001* (-combined_df['P_grid_neg'])).sum()
print('Original revenue of a day (unit_prod_price * P_grid_neg):', origin_revenue)

origin_load_cost = 0.5*(0.001*(combined_df['P_grid_pos']) * combined_df['unit_load_cost']).sum()
print('Original cost of a day (P_grid_pos) * unit_load_cost:', origin_load_cost)

original_cost = origin_load_cost - origin_revenue
print('Original Total cost:', original_cost)

In [ ]:
import matplotlib.pyplot as plt

# Bar chart for total cost comparison
costs = [original_cost, optimal_cost]
labels = ['Original', 'Optimized']
colors = ['blue', 'orange']

plt.figure(figsize=(6, 5))
plt.bar(labels, costs, color=colors)
plt.ylabel('Total Cost (€)')
plt.title('Total Cost Comparison')
for i, v in enumerate(costs):
    plt.text(i, v + 0.01 * max(costs), f"{v:.2f} €", ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Create a DataFrame for tabular comparison
comparison_table = pd.DataFrame({
    'Scenario': ['Original', 'Optimized'],
    'Total Cost (€)': [original_cost, optimal_cost]
})
comparison_table['Savings (€)'] = comparison_table['Total Cost (€)'].iloc[0] - comparison_table['Total Cost (€)']
comparison_table['Savings (%)'] = 100 * comparison_table['Savings (€)'] / comparison_table['Total Cost (€)'].iloc[0]
display(comparison_table)

In [ ]:
# plt.figure(figsize=(6, 6))
# plt.pie([optimal_cost, original_cost - optimal_cost], labels=['Optimized Cost', 'Savings'],
#         colors=['orange', 'green'], autopct='%1.1f%%', startangle=90)
# plt.title('Proportion of Savings in Total Original Cost')
# plt.show()

In [ ]:
savings_percent = (original_cost - optimal_cost) / original_cost * 100
print('savings percent:', savings_percent,'%')

In [ ]:
# Prepare original deferrable schedule
origin_df = selected_df[['time', 'ac_w', 'water_w']].copy()
origin_df = origin_df.rename(columns={'ac_w': 'Origin_P_deferrable0', 'water_w': 'Origin_P_deferrable1'})

# Prepare optimized deferrable schedule
opt_df = optimal_df[['P_deferrable0', 'P_deferrable1']].copy()
opt_df = opt_df.rename(columns={'P_deferrable0': 'Opt_P_deferrable0', 'P_deferrable1': 'Opt_P_deferrable1'})
origin_df['time_of_day'] = origin_df['time'].dt.time
comparison_df = origin_df[['time_of_day', 'Origin_P_deferrable0', 'Origin_P_deferrable1']].copy()
comparison_df['Opt_P_deferrable0'] = opt_df['Opt_P_deferrable0'].values
comparison_df['Opt_P_deferrable1'] = opt_df['Opt_P_deferrable1'].values
comparison_df = comparison_df[
    ['time_of_day', 
     'Origin_P_deferrable0', 'Opt_P_deferrable0', 
     'Origin_P_deferrable1', 'Opt_P_deferrable1']
]

comparison_df.head(24)

In [ ]:
comparison_df.to_csv('comparison_df.csv', index=False)

In [ ]:
# Calculate how many hours deferrable 0 and 1 are being used (nonzero) in both original and optimized schedules
# Each slot is 0.5 hour

def count_hours_used(power_series):
    # Count number of slots where power > 0
    slots_on = (power_series > 0).sum()
    return slots_on * 0.5

origin_hours_0 = count_hours_used(comparison_df['Origin_P_deferrable0'])
origin_hours_1 = count_hours_used(comparison_df['Origin_P_deferrable1'])
opt_hours_0 = count_hours_used(comparison_df['Opt_P_deferrable0'])
opt_hours_1 = count_hours_used(comparison_df['Opt_P_deferrable1'])

print(f"Original Deferrable 0 used: {origin_hours_0} hours")
print(f"Original Deferrable 1 used: {origin_hours_1} hours")
print(f"Optimized Deferrable 0 used: {opt_hours_0} hours")
print(f"Optimized Deferrable 1 used: {opt_hours_1} hours")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
optimal_df['timestamp'] = pd.to_datetime(optimal_df['timestamp'])
optimal_df['hour_decimal'] = optimal_df['timestamp'].dt.hour + optimal_df['timestamp'].dt.minute / 60
df_corr = pd.DataFrame({
    'Opt_P_deferrable0': optimal_df['P_deferrable0'],
    'Opt_P_deferrable1': optimal_df['P_deferrable1'],
    'P_PV': optimal_df['P_PV'],
    'P_Load': optimal_df['P_Load'],
    'time': optimal_df['hour_decimal']
})

df_corr.head()
# Correlation matrix
corr = df_corr.corr()
plt.figure(figsize=(7,5))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation: Optimal Schedule vs PV, Load, Time')
plt.show()


In [ ]:
import plotly.graph_objs as go

# Prepare time axis (use time_of_day if available, else fallback to time or range)
if 'time_of_day' in comparison_df.columns:
    time_axis = comparison_df['time_of_day'].astype(str)
elif 'time' in comparison_df.columns:
    time_axis = pd.to_datetime(comparison_df['time']).dt.strftime('%H:%M:%S')
else:
    time_axis = pd.date_range(start='2025-07-06 11:00:00', periods=len(comparison_df), freq='30min').strftime('%H:%M:%S')

fig = go.Figure()

# Bar for original deferrable device 0 (e.g., AC)
fig.add_trace(go.Bar(
    x=time_axis,
    y=comparison_df['Origin_P_deferrable0'],
    name='Original deferrable0',
    marker_color='blue',
    hovertemplate='Time: %{x}<br>Power: %{y:.0f} W'
))
# Bar for original deferrable device 1 (e.g., Water Heater)
fig.add_trace(go.Bar(
    x=time_axis,
    y=comparison_df['Origin_P_deferrable1'],
    name='Original deferrable1',
    marker_color='lightblue',
    hovertemplate='Time: %{x}<br>Power: %{y:.0f} W'
))
# Bar for optimized deferrable device 0 (e.g., AC)
fig.add_trace(go.Bar(
    x=time_axis,
    y=comparison_df['Opt_P_deferrable0'],
    name='Optimized deferrable0',
    marker_color='orange',
    hovertemplate='Time: %{x}<br>Power: %{y:.0f} W'
))
# Bar for optimized deferrable device 1 (e.g., Water Heater)
fig.add_trace(go.Bar(
    x=time_axis,
    y=comparison_df['Opt_P_deferrable1'],
    name='Optimized deferrable1',
    marker_color='gold',
    hovertemplate='Time: %{x}<br>Power: %{y:.0f} W'
))

fig.update_layout(
    barmode='group',
    title='Deferrable Devices Schedule: Original vs Optimized',
    xaxis_title='Time (Hour:Minute:Second)',
    yaxis_title='Power (W)',
    xaxis=dict(
        tickangle=45,
        rangeslider_visible=True
    ),
    hovermode='x unified'
 )


In [ ]:
import matplotlib.pyplot as plt

# Compute original (naive) cost per 30min slot
original_cost_per_slot =  (0.5 * 0.001 * combined_df['P_grid_pos'] * combined_df['unit_load_cost']  # import cost
    - 0.5 * 0.001 * (-combined_df['P_grid_neg']) * combined_df['unit_prod_price']  # export revenue
)
print("Total original (naive) cost:", original_cost_per_slot.sum())
# Get optimized cost per 30min slot
# If cost_fun_selfcons is profit (negative for cost), flip the sign
optimized_cost_per_slot = -optimal_df['cost_fun_selfcons'] if 'cost_fun_selfcons' in optimal_df.columns else None
print("Original cost per slot:", original_cost_per_slot.head())
print("Optimized cost per slot:", optimized_cost_per_slot.head())
print("Total optimal cost:", optimized_cost_per_slot.sum())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare time axis (use combined_df['time'] if available, else create a range)
if 'time' in combined_df.columns:
    time_axis = pd.to_datetime(combined_df['time'])
else:
    time_axis = pd.date_range(start='2025-07-06 11:00:00', periods=len(original_cost_per_slot), freq='30min')

# Set up bar width and positions
bar_width = 0.4
x = np.arange(len(time_axis))

plt.figure(figsize=(14, 5))
plt.bar(x - bar_width/2, original_cost_per_slot, width=bar_width, label='Naive (Original)', color='blue')
if optimized_cost_per_slot is not None:
    plt.bar(x + bar_width/2, optimized_cost_per_slot, width=bar_width, label='Optimized', color='orange')

# Set x-ticks to every 0.5 hour (every slot)
plt.xticks(
    ticks=x[::1],  # every slot (0.5h)
    labels=[t.strftime('%H:%M') for t in time_axis][::1],
    rotation=45
)

plt.xlabel('Time (Hour:Minute)')
plt.ylabel('Cost (€/30min)')
plt.title('Cost per Time Slot: Naive vs Optimized (Bar Chart)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objs as go

# Prepare time axis (ensure it's datetime)
if 'time' in combined_df.columns:
    time_axis = pd.to_datetime(combined_df['time'])
else:
    time_axis = pd.date_range(start='2025-07-06 11:00:00', periods=len(original_cost_per_slot), freq='30min')

fig = go.Figure()

# Bar for Naive (Original)
fig.add_trace(go.Bar(
    x=time_axis,
    y=original_cost_per_slot,
    name='Naive (Original)',
    marker_color='blue',
    hovertemplate='Time: %{x|%Y-%m-%d %H:%M}<br>Cost: %{y:.4f} €'
))

# Bar for Optimized
if optimized_cost_per_slot is not None:
    fig.add_trace(go.Bar(
        x=time_axis,
        y=optimized_cost_per_slot,
        name='Optimized',
        marker_color='orange',
        hovertemplate='Time: %{x|%Y-%m-%d %H:%M}<br>Cost: %{y:.4f} €'
    ))

fig.update_layout(
    barmode='group',
    title='Cost per Time Slot: Naive vs Optimized (Bar Chart)',
    xaxis_title='Time (Hour)',
    yaxis_title='Cost (€/30min)',
    xaxis=dict(
        tickformat='%H:%M',
        tickangle=45,
        rangeslider_visible=True
    ),
    hovermode='x unified'
)


In [ ]:
time_axis = comparison_df['time_of_day'].astype(str)
cumulative_original = original_cost_per_slot.cumsum()
cumulative_optimized = optimized_cost_per_slot.cumsum()

import numpy as np
import matplotlib.pyplot as plt

x = np.arange(len(time_axis))
plt.figure(figsize=(14, 5))
plt.plot(x, cumulative_original, label='Naive (Original)', marker='o', color='blue')
if cumulative_optimized is not None:
    plt.plot(x, cumulative_optimized, label='Optimized', marker='x', color='orange')
plt.xlabel('Time of Day (Hour:Minute:Second)')
plt.ylabel('Cumulative Cost (€)')
plt.title('Cumulative Cost Over the Day: Naive vs Optimized')
plt.legend()
plt.xticks(ticks=x, labels=time_axis, rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import plotly.graph_objs as go
import numpy as np

# Use only time_of_day as the x-axis (string format for clarity)
time_axis = comparison_df['time_of_day'].astype(str)
cumulative_original = np.cumsum(original_cost_per_slot)
cumulative_optimized = np.cumsum(optimized_cost_per_slot)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_axis,
    y=cumulative_original,
    mode='lines+markers',
    name='Naive (Original)',
    line=dict(color='blue'),
    marker=dict(symbol='circle', size=7)
))
fig.add_trace(go.Scatter(
    x=time_axis,
    y=cumulative_optimized,
    mode='lines+markers',
    name='Optimized',
    line=dict(color='orange'),
    marker=dict(symbol='x', size=7)
))
fig.update_layout(
    title='Cumulative Cost Over the Day: Naive vs Optimized (Interactive)',
    xaxis_title='Time of Day (Hour:Minute:Second)',
    yaxis_title='Cumulative Cost (€)',
    xaxis=dict(tickangle=45),
    legend=dict(x=0.01, y=0.99),
    hovermode='x unified',
    margin=dict(l=40, r=20, t=40, b=80)
 )


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Total grid import/export for original (combined_df)
original_import_kwh = combined_df['P_grid_pos'].sum() * 0.5 / 1000
original_export_kwh = -combined_df['P_grid_neg'].sum() * 0.5 / 1000
original_grid_kwh = combined_df['P_grid'].sum() * 0.5 / 1000

# Ttotal grid import/export for optimal (optimal_df)
optimal_import_kwh = optimal_df['P_grid_pos'].sum() * 0.5 / 1000
optimal_export_kwh = -optimal_df['P_grid_neg'].sum() * 0.5 / 1000
optimal_grid_kwh = optimal_df['P_grid'].sum() * 0.5 / 1000

print('Original total import kWh:', original_import_kwh)
print('Original total export kWh:', original_export_kwh)
print('Original total grid usage kWh:', original_grid_kwh)
print('Optimal total import kWh:', optimal_import_kwh)
print('Optimal total export kWh:', optimal_export_kwh)
print('Optimal total grid usage kWh:', optimal_grid_kwh)

In [ ]:
# Prepare data for bar chart
labels = ['Original Import', 'Original Export', 'Optimal Import', 'Optimal Export']
values = [original_import_kwh, original_export_kwh, optimal_import_kwh, optimal_export_kwh]
colors = ['blue', 'lightblue', 'orange', 'gold']

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, values, color=colors)
plt.ylabel('Total Energy (kWh)')
plt.title('Total Grid Import and Export: Original vs Optimized')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01*max(values), f"{yval:.2f}", ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
savings_percent_energy = (original_grid_kwh - optimal_grid_kwh) / original_grid_kwh * 100
print('Savings in total grid usage energy:', savings_percent_energy, '%')

In [ ]:
savings_percent_import = (original_import_kwh - optimal_import_kwh) / original_import_kwh * 100
print('Savings in total import energy:', savings_percent_import, '%')

In [ ]:
# Line chart: Import power (W) per 30-min slot (Original vs Optimal)
import matplotlib.pyplot as plt
import numpy as np

# Prepare time axis
if 'time' in combined_df.columns:
    time_axis = pd.to_datetime(combined_df['time'])
else:
    time_axis = pd.date_range(start='2025-07-06 11:00:00', periods=len(combined_df), freq='30min')

plt.figure(figsize=(14, 5))
plt.plot(time_axis, combined_df['P_grid_pos'], label='Original Import', color='blue', marker='o')
plt.plot(time_axis, optimal_df['P_grid_pos'], label='Optimal Import', color='orange', marker='x')
plt.xlabel('Time')
plt.ylabel('Grid Import Power (W)')
plt.title('Grid Import Power per 30-min Slot: Original vs Optimized')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
import plotly.graph_objs as go

# Interactive line chart: Import power (W) per 30-min slot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_axis,
    y=combined_df['P_grid_pos'],
    mode='lines+markers',
    name='Original Import',
    line=dict(color='blue'),
    marker=dict(symbol='circle', size=7)
))
fig.add_trace(go.Scatter(
    x=time_axis,
    y=optimal_df['P_grid_pos'],
    mode='lines+markers',
    name='Optimal Import',
    line=dict(color='orange'),
    marker=dict(symbol='x', size=7)
))
fig.update_layout(
    title='Grid Import Power per 30-min Slot: Original vs Optimized (Interactive)',
    xaxis_title='Time',
    yaxis_title='Grid Import Power (W)',
    xaxis=dict(tickangle=45),
    legend=dict(x=0.01, y=0.99),
    hovermode='x unified',
    margin=dict(l=40, r=20, t=40, b=80)
)


In [ ]:
## Separate line chart for grid use (P_grid)
plt.figure(figsize=(14, 5))
plt.plot(time_axis, combined_df['P_grid'], label='Original Grid Use', color='green', marker='o')
plt.plot(time_axis, optimal_df['P_grid'], label='Optimal Grid Use', color='red', marker='x')
plt.xlabel('Time')
plt.ylabel('Grid Use Power (W)')
plt.title('Grid Use Power per 30-min Slot: Original vs Optimized')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
# Interactive line chart for grid use (P_grid)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=time_axis,
    y=combined_df['P_grid'],
    mode='lines+markers',
    name='Original Grid Use',
    line=dict(color='green'),
    marker=dict(symbol='circle', size=7)
))
fig2.add_trace(go.Scatter(
    x=time_axis,
    y=optimal_df['P_grid'],
    mode='lines+markers',
    name='Optimal Grid Use',
    line=dict(color='red'),
    marker=dict(symbol='x', size=7)
))
fig2.update_layout(
    title='Grid Use Power per 30-min Slot: Original vs Optimized (Interactive)',
    xaxis_title='Time',
    yaxis_title='Grid Use Power (W)',
    xaxis=dict(tickangle=45),
    legend=dict(x=0.01, y=0.99),
    hovermode='x unified',
    margin=dict(l=40, r=20, t=40, b=80)
)
fig2.show()